# Sample code snippet in slides

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoTokenizer
sent = "Transformers are surprisingly sample-efficient."

# WordPiece/BPE-style (default from checkpoint)
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
out = tok(sent, return_tensors="pt")

print(tok.convert_ids_to_tokens(out["input_ids"][0]))
print("save to github")


['[CLS]', 'transformers', 'are', 'surprisingly', 'sample', '-', 'efficient', '.', '[SEP]']
save to github


In [ ]:
import torch
import torch.nn as nn
vocab_size, d_model = 30000, 512
tok_emb = nn.Embedding(vocab_size, d_model) # learned lookup

# LM head (logits over vocabulary)
lm_head = nn.Linear(d_model, vocab_size, bias=False)

# optional weight tying
lm_head.weight = tok_emb.weight

print(lm_head.weight.shape)
print(tok_emb.weight.shape)

torch.Size([30000, 512])
torch.Size([30000, 512])


Token embedding code

In [3]:
import torch
import torch.nn as nn
vocab_size, d_model = 30000, 512
tok_emb = nn.Embedding(vocab_size, d_model) # learned lookup
print("token emb mat", tok_emb.weight)

# LM head (logits over vocabulary)
lm_head = nn.Linear(d_model, vocab_size, bias=False)
print(lm_head.weight)

# optional weight tying
lm_head.weight = tok_emb.weight

print(lm_head.weight)
print(tok_emb.weight.shape)

token emb mat Parameter containing:
tensor([[-0.9509, -0.6686, -0.9180,  ...,  0.6268,  0.0182, -0.2016],
        [-0.5965,  1.8176,  0.5935,  ..., -0.6369, -0.8220, -0.5207],
        [-0.4496,  0.7260,  0.0735,  ..., -0.5989,  1.5681,  0.5585],
        ...,
        [ 2.2085,  3.0639, -0.3983,  ..., -0.6801, -0.0218,  0.5621],
        [ 0.2293, -0.3722,  0.9419,  ...,  0.2732, -2.1951, -1.1103],
        [-0.2046, -0.6572,  0.6872,  ...,  1.3169,  0.3338, -0.9886]],
       requires_grad=True)
Parameter containing:
tensor([[-0.0341, -0.0241,  0.0179,  ...,  0.0373,  0.0432, -0.0164],
        [ 0.0306, -0.0160, -0.0027,  ...,  0.0406, -0.0273,  0.0376],
        [-0.0123, -0.0291, -0.0113,  ..., -0.0257,  0.0147, -0.0130],
        ...,
        [ 0.0140, -0.0089, -0.0096,  ..., -0.0441, -0.0334,  0.0315],
        [-0.0101, -0.0353,  0.0218,  ...,  0.0026, -0.0140,  0.0288],
        [ 0.0256,  0.0361, -0.0401,  ..., -0.0325,  0.0231, -0.0101]],
       requires_grad=True)
Parameter containing

Sinusoidal pos emb

In [4]:
import torch
import math
def sinusoidal_pe(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(0, max_len).unsqueeze(1)

    div = torch.exp(
    torch.arange(0, d_model, 2) *
    (-math.log(10000.0) / d_model)
    )

    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)

    return pe # (max_len, d_model)

pe = sinusoidal_pe(512, 768)
print(pe.shape)
print(pe)

torch.Size([512, 768])
tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
          0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  8.2843e-01,  ...,  1.0000e+00,
          1.0243e-04,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  9.2799e-01,  ...,  1.0000e+00,
          2.0486e-04,  1.0000e+00],
        ...,
        [ 6.1950e-02,  9.9808e-01,  5.3552e-01,  ...,  9.9857e-01,
          5.2112e-02,  9.9864e-01],
        [ 8.7333e-01,  4.8714e-01,  9.9957e-01,  ...,  9.9857e-01,
          5.2214e-02,  9.9864e-01],
        [ 8.8177e-01, -4.7168e-01,  5.8417e-01,  ...,  9.9856e-01,
          5.2317e-02,  9.9863e-01]])


learned token + position emb

In [6]:
import torch
import torch.nn as nn
B, n = 4, 128
vocab_size = 30000
d_model = 512
max_len = 512

tok_emb = nn.Embedding(vocab_size, d_model)
pos_emb = nn.Embedding(max_len, d_model)

token_ids = torch.randint(0, vocab_size, (B, n))
positions = torch.arange(n).unsqueeze(0).expand(B, n)
print(positions.shape)
print(token_ids.shape)

x = tok_emb(token_ids) + pos_emb(positions)
print(x.shape) # (B, n, d_model)


torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128, 512])


casual mask and padding mask

In [16]:
import torch
L = 6
causal = torch.triu(torch.ones(L, L), diagonal=1).bool()
mask = torch.zeros(L, L)
mask = mask.masked_fill(causal, float("-inf"))
print(mask)

tensor([[0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0.]])


In [17]:
import torch
# token ids, PAD = 0
tokens = torch.tensor([
[5, 8, 9, 0, 0],
[3, 4, 7, 2, 0],
])
pad_mask = (tokens == 0) # (B, n), True means PAD

# For attention scores (B, h, n, n), expand to key dimension:
attn_mask = pad_mask[:, None, None, :] # (B,1,1,n)

print(attn_mask)

tensor([[[[False, False, False,  True,  True]]],


        [[[False, False, False, False,  True]]]])


In [23]:
import torch
import torch.nn as nn

B, n, vocab = 2, 5, 10000
logits = torch.randn(B, n, vocab)
targets = torch.randint(0, vocab, (B, n))

print(logits)
print(targets.shape)

PAD_ID = 0

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

loss = criterion(
logits.reshape(B * n, vocab),
targets.reshape(B * n)
)

print(loss)

tensor([[[-0.4477,  0.8268, -0.2230,  ...,  0.1767, -0.1170, -0.5906],
         [ 0.5331,  0.9272, -2.3988,  ..., -1.0404,  2.2106, -0.0967],
         [-0.3880, -1.1643,  0.1274,  ...,  0.8355,  0.1946,  2.6378],
         [-0.9256, -0.7683,  0.6890,  ..., -1.0234, -0.0194, -0.0454],
         [-0.0414, -1.8216, -1.3416,  ..., -0.2259,  0.4632, -0.7349]],

        [[ 0.7286,  0.5554, -0.2616,  ...,  0.3308, -0.9052,  1.3348],
         [-1.1698, -0.0341, -0.0504,  ..., -0.3570,  1.0023, -0.1656],
         [ 0.1174,  0.5098,  1.2033,  ..., -0.2271, -0.7518,  0.0880],
         [ 0.7647,  1.0955,  0.7279,  ..., -0.4914, -1.2090, -0.5664],
         [-0.7428,  1.0627,  0.9588,  ..., -0.9891, -1.2195,  0.4076]]])
torch.Size([2, 5])
tensor(10.0806)


MHSA and encoder stack

In [32]:
import torch
import torch.nn as nn

# Core MHSA module
mhsa = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True)

x = torch.randn(4, 128, 512) # (B, n, d)
y, attn = mhsa(x, x, x, need_weights=True)

print(y.shape)
print(attn.shape)

# Encoder layer + stacked encoder
enc_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, dim_feedforward=2048, batch_first=True)
print(enc_layer, '\n')

encoder = nn.TransformerEncoder(enc_layer, num_layers=6)
print(encoder)

z = encoder(x)

print(z.shape)

torch.Size([4, 128, 512])
torch.Size([4, 128, 128])
TransformerEncoderLayer(
  (self_attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
  )
  (linear1): Linear(in_features=512, out_features=2048, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (linear2): Linear(in_features=2048, out_features=512, bias=True)
  (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (dropout1): Dropout(p=0.1, inplace=False)
  (dropout2): Dropout(p=0.1, inplace=False)
) 

TransformerEncoder(
  (layers): ModuleList(
    (0-5): 6 x TransformerEncoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
      )
      (linear1): Linear(in_features=512, out_features=2048, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (linear2): Linear(in_features=2048, ou

Keras style (do not have one universal TransformerEncoder) -> build encoder blocks from layers

In [35]:
import keras
from keras import layers
inp = keras.Input(shape=(128, 512)) # (n, d) per sample

# Self-attention sublayer
x1 = layers.LayerNormalization()(inp)
a = layers.MultiHeadAttention(num_heads=8, key_dim=64)(x1, x1)
print(a)
x2 = layers.Add()([inp, a])

# FFN sublayer
x3 = layers.LayerNormalization()(x2)
f = layers.Dense(2048, activation="gelu")(x3)
f = layers.Dense(512)(f)

out = layers.Add()([x2, f])
print(out)

model = keras.Model(inp, out)

print(model)

<KerasTensor shape=(None, 128, 512), dtype=float32, sparse=False, ragged=False, name=keras_tensor_18>
<KerasTensor shape=(None, 128, 512), dtype=float32, sparse=False, ragged=False, name=keras_tensor_23>
<Functional name=functional_2, built=True>


# Transformer Families
Bản chất đều dùng attetion, chỉ khác là dùng phương pháp masking gì

In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ckpt = "bert-base-uncased" # mo hinh BERT nguyen ban
tok = AutoTokenizer.from_pretrained(ckpt)

model = AutoModelForSequenceClassification.from_pretrained(
ckpt,
num_labels=3)

batch = tok(
  ["This movie is excellent."],
  padding=True,
  truncation=True,
  return_tensors="pt"
)

out = model(**batch)
print(out.logits.shape) # (B, num_labels)
print(out)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 3])
SequenceClassifierOutput(loss=None, logits=tensor([[ 0.3183, -0.3559, -0.0500]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


In [14]:
import torch
from transformers import AutoTokenizer, AutoModel

ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(ckpt)
model = AutoModel.from_pretrained(ckpt) #AutoModel là mô hình gốc chỉ làm nhiệm  vụ trích xuất đặc trưng

text = "This movie is excellent."

inputs = tokenizer(text, return_tensors="pt")

print("Token IDs:", inputs["input_ids"][0].tolist())
print("Chuyển ID ngược về chữ:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))


with torch.no_grad():
    outputs = model(**inputs)


last_hidden_state = outputs.last_hidden_state
print("Kích thước ma trận tổng (Batch, Seq_len, Hidden_size):", last_hidden_state.shape)

cls_vector = last_hidden_state[:, 0, :]

print("Kích thước vector [CLS]:", cls_vector.shape) # [1, 768]
print("5 giá trị đầu tiên của vector [CLS]:\n", cls_vector[0, :5])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Token IDs: [101, 2023, 3185, 2003, 6581, 1012, 102]
Chuyển ID ngược về chữ: ['[CLS]', 'this', 'movie', 'is', 'excellent', '.', '[SEP]']
Kích thước ma trận tổng (Batch, Seq_len, Hidden_size): torch.Size([1, 7, 768])
Kích thước vector [CLS]: torch.Size([1, 768])
5 giá trị đầu tiên của vector [CLS]:
 tensor([ 0.1543, -0.1944,  0.1096, -0.2805, -0.2638])


## GPT-style generation

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM

ckpt = "gpt2"
tok = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForCausalLM.from_pretrained(ckpt)

inputs = tok("Deep learning is", return_tensors="pt")

out = model.generate(
  **inputs,
  max_new_tokens=50,
  do_sample=True,
  top_p=0.9,
  temperature=0.7
)

print(tok.decode(out[0], skip_special_tokens=True))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Deep learning is the art of making complex systems more robust by reducing the amount of data that must be stored on the system.

One of the most important principles of this approach is to reduce the amount of data that is stored on the system. This reduction is


## T5 Example: Text-to-Text

In [27]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

ckpt = "t5-small"
tok = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(ckpt)

text = "translate English to German: Thank you."
inputs = tok(text, return_tensors="pt")

out = model.generate(**inputs, max_new_tokens=40)

print(tok.decode(out[0], skip_special_tokens=True))


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Vielen Dank.


## ViT
ViT trade inductive bias of CNN for more flexible global attention

In [35]:
import torch
import torch.nn as nn

B, C, H, W = 4, 3, 224, 224
P = 16
d_model = 768

patch_embed = nn.Conv2d(
  in_channels=C,
  out_channels=d_model,
  kernel_size=P,
  stride=P
)

x = torch.randn(B, C, H, W)
print(x.shape)

z = patch_embed(x) # (B, d, 14, 14)
print(z.shape)

z = z.flatten(2).transpose(1, 2) # (B, 196, d)
print(z.shape)


torch.Size([4, 3, 224, 224])
torch.Size([4, 768, 14, 14])
torch.Size([4, 196, 768])


In [38]:
import timm
import torch

num_classes = 1000
model = timm.create_model(
"vit_base_patch16_224",
pretrained=True,
num_classes=num_classes,
)

data_cfg = timm.data.resolve_model_data_config(model)

print(data_cfg)

{'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'crop_pct': 0.9, 'crop_mode': 'center'}


## Pratical Note

In [7]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

num_classes = 5

ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForSequenceClassification.from_pretrained(
  ckpt,
  num_labels=num_classes,
)

batch = tokenizer(
  ["A good example.", "A bad example."],
  padding=True,
  truncation=True,
  return_tensors="pt",
)

print(batch)

out = model(**batch)

print(out)
print(out.logits.shape) # (B, num_classes)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'input_ids': tensor([[ 101, 1037, 2204, 2742, 1012,  102],
        [ 101, 1037, 2919, 2742, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1]])}
SequenceClassifierOutput(loss=None, logits=tensor([[-0.0683,  0.7016, -0.1585,  0.2119,  0.5265],
        [-0.0666,  0.7232, -0.1612,  0.2384,  0.5402]],
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)
torch.Size([2, 5])


# Test

In [15]:
a = torch.arange(3)
print(a)
print(a.shape, '\n')

b = a.unsqueeze(1)
print(b)
print(b.shape, '\n')

a.unsqueeze_(0)
print(a)
print(a.shape)


tensor([0, 1, 2])
torch.Size([3]) 

tensor([[0],
        [1],
        [2]])
torch.Size([3, 1]) 

tensor([[0, 1, 2]])
torch.Size([1, 3])
